### Autors

Ziyi Hong - *2694933*

Danylo Tulainov - *5070969*

# Laborarbeit Teil 1

## Datensatz

In diesem Notebook werden Sie selbstständig mit dem Datensatz `passengers-dataset.csv` arbeiten.

Der Datensatz enthält Passagierdaten eines historischen Schiffsunglücks. Das sind die Spalten der Datei:
* PassengerID: Passagier-ID
* Survived: Indikator ob Passagier überlebt hat (1=ja, 0=nein)
* Pclass: Ticketklasse 
* Name: Name des Passagiers
* Sex: Geschlecht des Passagiers
* Age: Alter des Passagiers
* SibSp: Zahl der Geschwister und Ehepartner eines Passagiers an Bord
* Parch: Zahl der Elternteile und Kinder eines Passagiers an Bord
* Ticket: Ticket-ID
* Fare: Ticketpreis
* Cabin: Kabine des Passagiers
* Embarked: Zustiegshafen (C=Cherbourg, Frankreich; Q=Queenstown, Irland; S=Southampton, Großbritannien)

## Aufgabe

**Ihre Aufgabe ist es ein möglichst gutes Modell zu entwickeln, das vorhersagt, ob ein Passagier überlebt.**

## Hinweise

Die Aufgabenstellung ist bewusst offen gehalten. Berücksichtigen Sie bei der Bearbeitung der Aufgabe die folgenden Hinweise:
* Dokumentieren Sie Ihr Vorgehen strukturiert und nachvollziehbar.
* Begründen Sie Ihre Vorgehensweise und Entscheidungen.
* Interpretieren Sie Ergebnisse.
* Erstellen Sie, wo möglich und sinnvoll, Plots.
* **Nutzen Sie nur Methoden, Konzepte und Begründungen, die in der Vorlesung behandelt wurden. Falls dies nicht möglich ist, begründen Sie warum.**
* Identifizieren Sie Schwachstellen/Limitationen Ihrer Arbeit und erläutern Sie diese auf Basis der Vorlesungsinhalte.

Berücksichtigen Sie die folgenden Hinweise für die Abgabe:
* Arbeiten Sie in 2er-Gruppen und geben Sie je Gruppe ein Notebook ab.
* Nennen Sie im abgegebenen Notebook die Gruppenmitglieder.
* **Für die Verwendung von KI berücksichtigen Sie die Hinweise von Herrn Prof. Hänisch.**
* Die erste Abgabe muss am Dienstag, den 17.03., zwischen 15:30 und 16:15 Uhr per Moodle erfolgen.
* Die endgültige Abgabe muss bis Dienstag, den 24.03., um 23:59 Uhr per Moodle erfolgen.
* Gewertet wird die endgültige Abgabe. Wenn Sie zwischen erster und endgültiger Abgabe Änderungen vornehmen, müssen Sie diese begründen.

## Data Understanding

The first step of creating the machine learning model - according to **Crisp-DM** - is *Data Understanding*. (Actually the first step is *Business Understanding* but it was decided to omit this stage)

At this stage the following work will be done:
- Load of dataset
- Drop of non-relevant features
- Store the target feature to `y` and training features to `X`
- Feature Engineering:
  - Count the missing values
  - Find numerical and categorical features
  - `One-Hot-Encoding` for categorical
  - Find correlating features
  - Outline top features


#### Import packages

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score


In [ ]:
# load data into dataframe
file_path = 'passengers-dataset.csv'
df = pd.read_csv(file_path)
df.head()

`df.info()` helps to see the features of dataset

In [ ]:
df.info()

#### Drop the columns

It was decided that `PassengerId`, `Name`, and `Ticket` are irrelevant to the target value.

In [ ]:
df = df.drop(columns=["PassengerId", "Name", "Ticket"], errors="ignore")
df.head()

#### Separation of features from target

This step is made to separate training features from target.
Target is saved to `y`, training to `X`.

In [ ]:
y = df["Survived"]
X = df.drop(columns=["Survived"])

# number of examples and features
X.shape

#### Find missing values

The next step is crucial in *feature engineering*.
If the model is trained with missing values, its results will interfere with the analysis and final predictions.

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
missing_counts

#### Find numerical data and categorical data

Another important step of *feature engineering*.

In [ ]:
# numerical
numerical_data = X.select_dtypes("number")
numerical_data.info()

In [ ]:
# categorical
string_data = X.select_dtypes(object)
string_data.info()

#### Find the unique categories per categorical features

In [ ]:
num_categories = string_data.nunique().sort_values(ascending=False)
num_categories

#### Drop the `cabin` feature:

`cabin` feature is the most problematic feature in dataset. Not only it has 147 unique categories, 687 rows are `null`. To prevent the learning process from this feature -> it was decided to drop it from dataset.

In [ ]:
df_clean = df.drop(columns=["Cabin"], errors="ignore")
df_clean.head()

#### `One-hot-encoding`

Categorical features are transformed via **One-hot-encoding** and `get_dummies()` method. This step ensures that dataset is properly formatted for correlation analysis.

In [ ]:
df_encoded = pd.get_dummies(df_clean, drop_first=True)
df_encoded

#### Corrrelation

Correlation helps identify relationships between features and find out which features are most influential for the target variable `y`.

In [ ]:
# correlation matrix
corr_matrix = df_encoded.corr()

# correlation with target variable
corr_with_target = corr_matrix["Survived"].sort_values(ascending=False)
top_features = corr_with_target[1:10]
top_features

#### Find the top features

Find the top features and check whether each one belongs to numerical or categorical. It is a validation step to confirm the nature of the most significant features.

In [ ]:
# names of top features
top_feature_names = top_features.index

# numerical columns in X
numeric_cols = numerical_data.columns

# check: numerical or categorical
for feature in top_feature_names:
    if feature in numeric_cols:
        print(f"{feature}: numerical")
    else:
        print(f"{feature}: categorical")

But the top features might not be useful, because it is from postive to negative. It doesn't mean negative features are not important. We decided to see the |correlation| (the absolute)

## Data preparation

The next stage of **CRISP-DM** is *Data Preparation*.

This stage consists of following steps:
- Selecting features for training
- One hot encoding of categorical features
- Find missing values in selected features
- Replacing missing values
- Division of data into training and testing sets
- Feature scaling

Features selected and reasons:
- Sex_male: strongest correlation with survival −0.54.
- Pclass: −0.34 correlation.
- Fare: positive correlation 0.26.
- Age	has weak correlation but we still decided to include it. Because younger one might have better chance to live.
- Embarked_S: −0.16.

In [ ]:
selected_features = [
    "Sex_male",
    "Pclass",
    "Fare",
    "Age",
    "Embarked_S",
]


#### One hot encoding

At the *Data Understanding* stage `One-hot-encoding` was made to the whole dataframe for the sake of finding correlation.
Here, at this stage, it is done for categorical features in `X`which will be used for training of the model.

In [ ]:
# one-Hot-Encoding categorical features
X = pd.get_dummies(X, drop_first=True)
X

#### Load selected features into `X`

In [ ]:
# select
X = X[selected_features]
X.head()

#### Find and replace missing values

The next step's code finds and replaces missing values in chosen features.
Replacement happens in feature `Age` with **median** value.

In [ ]:
# find missing
missing_counts = X.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
missing_counts

# Replace missing values in age with median
print(X['Age'].median())

X = X.copy()
X["Age"] = X["Age"].fillna(X['Age'].median())

X.isna().sum()


#### Train-test split

The data is prepared for learning. In the following code the data is splitted to train and test in percentage 80/20.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape)

## Modeling

The next stage in **CRISP-DM** is *Modeling*.

This stage consists of following steps:
- Model comparison using pipelines
- Hyperparameter tuning
- Model evaluation
- Model training
- Model scaling
- Performance assessment

#### Model Comparison using pipelines

In this code snippet multiple `Pipeline` objects are defined to streamline and model execution. Each pipeline bundles feature engineering steps directly with a specific classifier like `Logistic Regression`, `SVC`, `MLPClassifier`.

Cross-validation is performed using `StratifieldKFold`. Process  calculates the mean accuracy and standartd deviation for each model, which serves as the basis for selecting __the most effective algorithm__.

In [ ]:
# compare models and choose the best models using scikit learn

pipe_log = Pipeline([
    ("poly", PolynomialFeatures(degree=2)),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(penalty="l2", max_iter=1000))
])


pipe_svc = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42, max_iter=1000))
])

pipe_mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(hidden_layer_sizes=10, random_state=42, max_iter=1000))
])

pipe_tree = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", DecisionTreeClassifier(random_state=42, max_depth=4))
])

# dictionary for all the pipelines
pipelines = {
    "log reg": pipe_log,
    "mlp": pipe_mlp,
    "svc": pipe_svc,
    "tree": pipe_tree,
}


# cross validation and compare the scores of those pipelines
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    print(f"{name}: {scores.mean()}, ±, {scores.std()}")


We choose to go on with decision tree as the first model.

## Decision Tree

In [ ]:
# svc and tree has better avergae accuracy

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report



tree_gini = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

tree_gini.fit(X_train, y_train)

print("Train accuracy:", accuracy_score(y_train, tree_gini.predict(X_train)))
print("Test accuracy :", accuracy_score(y_test, tree_gini.predict(X_test)))


print(classification_report(y_test, tree_gini.predict(X_test), target_names=["dead", "survived"]))

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(18, 8))
plot_tree(
    tree_gini,
    feature_names=X.columns,
    class_names=["dead", "survived"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
ConfusionMatrixDisplay.from_estimator(
    tree_gini,
    X_test,
    y_test,
    display_labels=["dead", "survived"]
)
plt.show()

In [ ]:
print(classification_report(y_test, tree_gini.predict(X_test), target_names=["dead", "survived"]))

To find the best depths, we *scaler* the features and split the dataset into  *train, validation, test*

In [ ]:
depths = range(1, 16)

train_scores = []
val_scores = []

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

# scalierung
scaler = StandardScaler()
X_train_sub = scaler.fit_transform(X_train_sub)
X_val = scaler.transform(X_val)

for depth in depths:
    tree_depth = DecisionTreeClassifier(
        criterion="gini",
        max_depth=depth,
        random_state=42
    )
    tree_depth.fit(X_train_sub, y_train_sub)

    train_scores.append(accuracy_score(y_train_sub, tree_depth.predict(X_train_sub)))
    val_scores.append(accuracy_score(y_val, tree_depth.predict(X_val)))

depth_results = pd.DataFrame({
    "max_depth": list(depths),
    "train_accuracy": train_scores,
    "validation_accuracy": val_scores
})

depth_results

In [ ]:
best_depth = depth_results.loc[depth_results["validation_accuracy"].idxmax(), "max_depth"]
best_val_accuracy = depth_results["validation_accuracy"].max()

print("Beste Validation-Accuracy bei max_depth =", best_depth)
print("Beste Validation-Accuracy:", round(best_val_accuracy, 4))

plt.figure(figsize=(8, 5))
plt.plot(depth_results["max_depth"], depth_results["train_accuracy"], marker="o", label="Train accuracy")
plt.plot(depth_results["max_depth"], depth_results["validation_accuracy"], marker="o", label="Validation accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: Accuracy in Abhängigkeit von der Baumtiefe")
plt.xticks(list(depths))
plt.grid(True)
plt.legend()
plt.show()


Here we use the best max_depth=3, and use *cross validation* to see the final test accuracy

In [ ]:
tree_tuned = DecisionTreeClassifier(
    criterion="gini",
    max_depth=best_depth,
    random_state=42
)
tree_tuned.fit(X_train, y_train)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(tree_tuned, X, y, cv=cv, scoring="accuracy")

print("Tree test mean accuracy:", scores.mean(), "±", scores.std())

### Decision Tree -> Random Forest

Since we used decision tree, we want to see whether random forest will achieve a better result.

This step defines a search space for model parameters and evaluates multiple configurations.

The loop trains a new `RandomForestClassifier` for every combination of `n_estimators` and `max_depth`.

In [ ]:
rf_results = []

n_estimators_list = [10, 50, 100]
max_depth_list = [3, 5, 10, None]

for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        rf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )
        rf.fit(X_train_sub, y_train_sub)

        train_acc = accuracy_score(y_train_sub, rf.predict(X_train_sub))
        val_acc = accuracy_score(y_val, rf.predict(X_val))

        rf_results.append({
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc
        })

rf_results = pd.DataFrame(rf_results)
rf_results

#### Selection of best parameters

Initialization of the final model instance. This step extracts the optimal values for the number of trees and maximum depth.

In [ ]:
best_rf_row = rf_results.loc[rf_results["val_accuracy"].idxmax()]
best_rf_row

In [ ]:
best_rf = RandomForestClassifier(
    n_estimators=int(best_rf_row["n_estimators"]),
    max_depth=int(best_rf_row["max_depth"]),
    random_state=42
)

#### Final model training and assessment

The final model is trained on the complete and scaled training set. Once it's ready, a calculation of the accuracy on the held-out test follows. The resulting value represents the final performance metric.

In [ ]:
best_rf.fit(X_train, y_train)

rf_test_accuracy = accuracy_score(y_test, best_rf.predict(X_test))
print("Test Accuracy of the best Random Forest:", round(rf_test_accuracy, 4))

## Evaluation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(best_rf,X,y,cv=cv,scoring="accuracy")

print("Accuracy pro Fold:", scores)

print("Mean Accuracy:", scores.mean(), "±", scores.std())

| Model | Mean Accuracy | Std Dev |
|---|---|---|
| **Random Forest** | **0.8361** | ±0.0084 |
| SVC (RBF kernel) | 0.8258 | ±0.0207 |
| Logistic Regression | 0.8202 | ±0.0194 |
| Decision Tree | 0.8146 | ±0.0173 |
| MLP | 0.8090 | ±0.0188 |

*Random Forest* achieves the highest mean accuracy and also the lowest standard deviation, making it both the most accurate and the most stable model across all folds.


# Limitation #

The best model: Random Forest has only 83% mean accuracy.

Why?
1. Small Dataset
2. Feature selection might be bad. There are only 5 features selected.
    - for example: Parch, SibSp can be a useful feature, which shows many children and family member they have.
3. 